# 🕵️ Dia 42 — Medindo atividade de sites de vazamento de ransomware sem coletar vítimas

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 42* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-42.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O conselho da Cooperativa Serra Azul recebeu um print alarmista: *"cooperativas são o alvo nº 1 do ransomware"*. Marta Quintão quer um número até sexta. Rafa avisa: nada de visitar site de vazamento, nada de anotar nome de vítima, e nada de misturar ransomware com o golpe do IscaKit, que é phishing.

Neste notebook você vai:

1. **Ingerir postagens com uma lista de permissão de campos**: o que identifica vítima nunca chega ao disco.
2. **Desenhar o mapa de calor grupo × mês só com Python puro** (sem pandas, sem matplotlib).
3. **Medir as ressalvas**: postagens sem data de ataque, derrubada, *rebrand*, contagens pequenas.
4. **Jogar o quiz "postagem ou ataque?"** e escrever a resposta de cinco linhas ao conselho.

🐍 Só biblioteca padrão. Os dados são **sintéticos** (grupos fictícios `grupo-alfa`… `grupo-eco`). Rode de cima para baixo.

## 1. Carregar as postagens sintéticas

O arquivo `caso-continuo/dados/dia-42/postagens-sinteticas.jsonl` do repositório tem o **mesmo formato** do `postings.jsonl` do original: uma linha JSON por postagem, com `group`, `month`, `attack_month`, `sector` e `country`.

A célula tenta baixá-lo do GitHub. Sem internet, ela **gera o mesmo arquivo aqui**, com o mesmo código e a mesma semente aleatória (o resultado é idêntico). É o plano B offline.

In [ ]:
import json                    # cada linha do arquivo é um objeto JSON
import urllib.request          # para tentar baixar o arquivo do repositório
import random                  # usado pelo gerador do plano B

URL = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/"
       "main/caso-continuo/dados/dia-42/postagens-sinteticas.jsonl")

# ---------- Plano B: o gerador sintético (idêntico ao que criou o arquivo) ----------
import random

GRUPOS_BASE = {          # média de postagens/mês (fictício)
    "grupo-alfa": 7,
    "grupo-bravo": 4,
    "grupo-charlie": 3,
    "grupo-delta": 9,    # "derrubado" em 2024-02
}
SETORES = ["Manufacturing", "Technology", "Healthcare", "Financial Services",
           "Education", "Retail", "Construction", "Government", "Unknown"]
PAISES = ["US", "US", "US", "DE", "GB", "BR", "BR", "IT", "FR", "CA", ""]


def meses(inicio=(2023, 1), fim=(2026, 9)):
    a, m = inicio
    while (a, m) <= fim:
        yield f"{a:04d}-{m:02d}"
        m += 1
        if m == 13:
            a, m = a + 1, 1


def mes_anterior(mes, n):
    a, m = int(mes[:4]), int(mes[5:])
    for _ in range(n):
        m -= 1
        if m == 0:
            a, m = a - 1, 12
    return f"{a:04d}-{m:02d}"


def gerar(semente=42):
    rnd = random.Random(semente)
    linhas = []
    for mes in meses():
        ativos = dict(GRUPOS_BASE)
        if mes >= "2024-02":
            ativos["grupo-delta"] = 1 if mes in ("2024-05", "2024-07") else 0   # resíduo pós-derrubada
        if mes >= "2024-06":
            ativos["grupo-eco"] = 6                                            # o "novo" nome
        if mes == "2025-03":
            ativos["grupo-bravo"] = 15                                         # pico (exploração em massa fictícia)
        for grupo, media in ativos.items():
            n = max(0, int(rnd.gauss(media * 0.6, 1.2))) if media else 0
            for _ in range(n):
                setor = rnd.choice(SETORES)
                pais = rnd.choice(PAISES)
                if pais == "BR" and mes >= "2025-10" and rnd.random() < 0.35:
                    setor = "Financial Services"                               # leve alta BR/financeiro recente
                ataque = mes_anterior(mes, rnd.choice([0, 1, 1, 2])) if rnd.random() < 0.55 else ""
                linhas.append({"group": grupo, "month": mes, "attack_month": ataque,
                               "sector": setor, "country": pais})
    return linhas
# ------------------------------------------------------------------------------------

try:
    # timeout curto: se a rede estiver lenta, não queremos travar a aula
    with urllib.request.urlopen(URL, timeout=10) as resposta:
        texto = resposta.read().decode("utf-8")
    POSTAGENS = [json.loads(l) for l in texto.splitlines() if l.strip()]
    origem = "repositório (GitHub)"
except Exception as erro:          # sem rede, 404, proxy... qualquer falha cai aqui
    POSTAGENS = gerar(semente=42)  # mesma semente = mesmos números do arquivo
    origem = f"gerador local (plano B: {type(erro).__name__})"

print(f"{len(POSTAGENS)} postagens carregadas de: {origem}")
print("Exemplo:", POSTAGENS[0])

## 2. A ingestão por lista de permissão

O original usa `jq` para jogar fora `victim`, `domain`, `description` e `url` **antes** de gravar. Aqui fazemos o mesmo em Python, com uma diferença de mentalidade:

- **Lista de bloqueio** (*denylist*): "apague estes campos". Se a API criar um campo novo amanhã (`victim_contact`, por exemplo), ele passa.
- **Lista de permissão** (*allowlist*): "copie só estes campos". O campo novo nunca entra.

Para testar, criamos um registro **simulado** no formato da API, com campos de vítima obviamente falsos (`VITIMA-FICTICIA`), e vemos o filtro trabalhar.

In [ ]:
# Os únicos campos que podem existir no nosso disco.
PERMITIDOS = ("group", "month", "attack_month", "sector", "country")

def limpar(registro_api: dict, grupo: str) -> dict:
    """Converte um registro no formato da API v2 do ransomware.live
    para o formato do postings.jsonl, mantendo SÓ os campos permitidos."""
    bruto = {
        "group": grupo,                                              # o slug que pedimos
        "month": (registro_api.get("discovered") or "")[:7],         # AAAA-MM em que o rastreador viu
        "attack_month": (registro_api.get("attackdate") or "")[:7],  # pode vir vazio: é uma ressalva!
        "sector": registro_api.get("activity") or "Unknown",         # atribuído pelo agregador
        "country": registro_api.get("country") or "",
    }
    # O "return" só copia o que está na lista. Qualquer outro campo morre aqui.
    return {campo: bruto[campo] for campo in PERMITIDOS}


# Um registro SIMULADO, com os campos perigosos preenchidos por marcadores falsos.
simulado = {
    "victim": "VITIMA-FICTICIA",              # nome da vítima: NUNCA pode passar
    "domain": "vitima-ficticia.example",      # domínio da vítima: idem
    "description": "texto do grupo criminoso",
    "url": "http://endereco-ficticio.invalid",
    "victim_contact": "campo novo inventado", # o campo que uma denylist deixaria passar
    "discovered": "2026-09-14 10:22:00",
    "attackdate": "",
    "activity": "Financial Services",
    "country": "BR",
}
limpo = limpar(simulado, "grupo-alfa")
print("Entrou :", sorted(simulado))
print("Saiu   :", limpo)

# O equivalente do `grep -c victim postings.jsonl` do checkpoint original:
linha = json.dumps(limpo)
print("Ocorrências de 'victim' na linha limpa:", linha.count("victim"))

In [ ]:
# Agora gravamos o nosso postings.jsonl (sintético) e fazemos o teste do checkpoint
# no arquivo inteiro, exatamente como o original pede.
with open("postings.jsonl", "w", encoding="utf-8") as f:
    for p in POSTAGENS:
        # Passamos de novo pela lista de permissão: defesa em profundidade.
        f.write(json.dumps({k: p[k] for k in PERMITIDOS}, ensure_ascii=False) + "\n")

with open("postings.jsonl", encoding="utf-8") as f:
    conteudo = f.read()

proibidos = ["victim", "domain", "description", "url"]
for termo in proibidos:
    print(f"grep -c {termo:<12} → {conteudo.count(termo)}")
assert all(conteudo.count(t) == 0 for t in proibidos), "Algum campo proibido vazou!"
print("✅ Nenhum campo de vítima no disco.")

## 3. O mapa de calor grupo × mês, só com texto

O original usa pandas + matplotlib. Aqui a ideia é a mesma (uma tabela dinâmica grupo × mês, colorida por intensidade), desenhada com os caracteres `·░▒▓█`. Cada coluna é um mês, de jan/2023 a set/2026. Repare em três coisas:

- a linha do `grupo-delta` **some** em fev/2024: é a nossa "Operação Cronos" simulada;
- o `grupo-eco` **nasce** em jun/2024: novo nome ou novos operadores? O gráfico não sabe;
- o `grupo-bravo` tem um **pico** isolado.

In [ ]:
from collections import Counter, defaultdict

# Tabela dinâmica: (grupo, mês) → número de postagens.
tabela = Counter((p["group"], p["month"]) for p in POSTAGENS)
grupos = sorted({p["group"] for p in POSTAGENS})
todos_meses = sorted({p["month"] for p in POSTAGENS})
maximo = max(tabela.values())

# Escala de "cor" em texto: quanto mais postagens, mais escuro.
ESCALA = " ·░▒▓█"
def bloco(n):
    if n == 0:
        return ESCALA[0]
    # distribui 1..maximo nos 5 níveis restantes
    return ESCALA[1 + min(4, (n - 1) * 5 // maximo)]

print("Postagens em sites de vazamento por mês e grupo (data de DESCOBERTA; dados SINTÉTICOS)")
print(" " * 15 + "".join("|" if m.endswith("-01") else " " for m in todos_meses) + "  ← '|' = janeiro")
for g in grupos:
    linha = "".join(bloco(tabela[(g, m)]) for m in todos_meses)
    total = sum(tabela[(g, m)] for m in todos_meses)
    print(f"{g:<14} {linha}  total={total}")
print(f"\nEscala: '{ESCALA[1]}'=pouco ... '{ESCALA[-1]}'={maximo} postagens. Período: {todos_meses[0]} a {todos_meses[-1]}.")

In [ ]:
# A tabela "dos últimos seis meses" que o heat.py do original imprime,
# para conferir o desenho contra os números.
ultimos = todos_meses[-6:]
print(f"{'grupo':<14}" + "".join(f"{m:>9}" for m in ultimos))
for g in grupos:
    print(f"{g:<14}" + "".join(f"{tabela[(g, m)]:>9}" for m in ultimos))

# O mês de pico de cada grupo: é o que o passo 3 do original manda anotar.
print("\nMês de pico por grupo:")
for g in grupos:
    mes_pico = max(todos_meses, key=lambda m: tabela[(g, m)])
    print(f"  {g:<14} {mes_pico}  ({tabela[(g, mes_pico)]} postagens) → explicação pública? se não houver: 'sem explicação'")

## 4. As ressalvas, medidas

O passo 3 do original pede a **proporção de postagens sem data de ataque** por grupo (o comando `jq -rs 'group_by(.group)...'`). Quanto maior essa proporção, mais o eixo "mês" significa só "quando o rastreador percebeu".

In [ ]:
por_grupo = defaultdict(list)
for p in POSTAGENS:
    por_grupo[p["group"]].append(p)

for g in grupos:
    sem_data = sum(1 for p in por_grupo[g] if p["attack_month"] == "")
    total = len(por_grupo[g])
    print(f"{g:<14} {sem_data}/{total} postagens sem data de ataque ({sem_data/total:.0%})")

## 5. ➕ A pergunta do conselho, em números

*"As postagens que mencionam o setor financeiro no Brasil aumentaram nos últimos 12 meses?"*

Comparamos os 12 meses mais recentes (out/2025 a set/2026) com os 12 anteriores. Preste atenção ao tamanho das contagens antes de olhar a porcentagem.

In [ ]:
br_fin = [p for p in POSTAGENS if p["country"] == "BR" and p["sector"] == "Financial Services"]

recentes = [p for p in br_fin if "2025-10" <= p["month"] <= "2026-09"]
anteriores = [p for p in br_fin if "2024-10" <= p["month"] <= "2025-09"]
a, b = len(anteriores), len(recentes)

print(f"Postagens BR + Financial Services, out/24–set/25: {a}")
print(f"Postagens BR + Financial Services, out/25–set/26: {b}")
if a:
    print(f"Variação: {b - a:+d} postagens ({(b - a) / a:+.0%})")

br_total = [p for p in POSTAGENS if p["country"] == "BR"]
sem_data_br = sum(1 for p in br_total if p["attack_month"] == "")
print(f"\nPostagens BR (todos os setores): {len(br_total)}; sem data de ataque: {sem_data_br} ({sem_data_br/len(br_total):.0%})")
print("País vazio (agregador não sabe):", sum(1 for p in POSTAGENS if p["country"] == ""), "postagens")
print("\n⚠️  Com contagens deste tamanho, uma única postagem a mais muda a porcentagem em dezenas de pontos.")

## 6. 🎮 Quiz: postagem ou ataque?

Dez frases que poderiam estar num relatório. Para cada uma, responda **S** se a frase é **defensável** com os dados deste dia, ou **N** se não é.

Quer só ler o gabarito comentado? Mude `MODO_JOGO` para `False`.

In [ ]:
# (frase, é_defensável, por quê)
FRASES = [
    ("O grupo-alfa fez 12 ataques em março.", False,
     "Postagem não é ataque. A data é da descoberta, e quem pagou não aparece."),
    ("O grupo-alfa teve 12 postagens descobertas pelo agregador em março.", True,
     "Descreve exatamente o que o dado mede."),
    ("O setor financeiro é o mais visado pelo ransomware no Brasil.", False,
     "Setor é atribuído automaticamente, quem paga some dos dados, e a contagem BR é pequena."),
    ("As postagens BR/financeiro passaram de 5 para 6 em 12 meses; com contagens tão pequenas, isso não indica tendência.", True,
     "Número absoluto + ressalva de amostra pequena."),
    ("O grupo-delta encerrou as atividades em fevereiro de 2024.", False,
     "Só sabemos que as postagens com esse nome pararam. Os operadores podem seguir com outro nome."),
    ("Postagens com o nome grupo-delta praticamente cessaram após fev/2024, mês da derrubada simulada; o grupo-eco surge em jun/2024, e não sabemos se há relação.", True,
     "Separa o observado da hipótese de rebrand."),
    ("A cooperativa do print encaminhado ao conselho foi invadida.", False,
     "É uma alegação de criminoso, não verificada, e está fora do escopo."),
    ("O Pescador está migrando do phishing para o ransomware.", False,
     "Nada nos dados liga o IscaKit a ransomware. É especulação sem evidência."),
    ("Aproximadamente metade das postagens não tem data de ataque, então o eixo de meses reflete a descoberta.", True,
     "É a ressalva medida no passo 4."),
    ("Uma cooperativa de Minas apareceu no site de vazamento X em 3/9; segue o nome para conferência.", False,
     "Nome de vítima não entra no trabalho de hoje. Ponto."),
]

MODO_JOGO = True   # False = só mostra o gabarito

acertos = 0
for i, (frase, ok, porque) in enumerate(FRASES, start=1):
    print(f"\n[{i:02d}/{len(FRASES)}] \"{frase}\"")
    if MODO_JOGO:
        resp = ""
        while resp not in ("S", "N"):                       # insiste até vir S ou N
            resp = input("   Defensável? (S/N): ").strip().upper()
        acertou = (resp == "S") == ok
        acertos += acertou
        print("   ✅ Isso." if acertou else "   ❌ Não.")
    print(f"   Gabarito: {'DEFENSÁVEL' if ok else 'NÃO DEFENSÁVEL'} — {porque}")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(FRASES)} — Checkpoint BR pede pelo menos 8.")

## 7. 📝 Sua resposta ao conselho (cinco linhas + uma)

Preencha o formulário e rode. A célula conta as linhas, procura palavras proibidas ("ataque", "ataques", "invadid") e grava `notas/dia42-resposta-conselho.md`.

In [ ]:
#@title Resposta ao conselho { display-mode: "form" }
linha1 = "Pergunta: as postagens em sites de vazamento que citam o setor financeiro no Brasil cresceram em 12 meses?"  #@param {type:"string"}
linha2 = "Dado: passaram de 5 para 6 postagens (out/24-set/25 vs. out/25-set/26), fonte agregada, dados sintéticos do exercício."  #@param {type:"string"}
linha3 = "Leitura: contagem pequena demais para indicar tendência; uma postagem a mais muda a porcentagem em 20 pontos."  #@param {type:"string"}
linha4 = "Ressalvas: postagens são alegações de criminosos; quem pagou não aparece; setor e país são atribuídos automaticamente."  #@param {type:"string"}
linha5 = "Ação: manter backups testados e o plano de resposta; não há dado que coloque a Serra Azul como alvo específico."  #@param {type:"string"}
linha6 = "O golpe do 'Pix bloqueado' (IscaKit) é phishing de credencial e não tem relação demonstrada com ransomware."  #@param {type:"string"}

from pathlib import Path

linhas = [linha1, linha2, linha3, linha4, linha5]
problemas = []
for n, l in enumerate(linhas, start=1):
    baixo = l.lower()
    for proibida in ("ataque", "invadid"):          # "ataque" pega "ataques" também
        if proibida in baixo:
            problemas.append(f"linha {n} usa '{proibida}…' — troque por 'postagem' ou 'vítima alegada'")
    if not l.strip():
        problemas.append(f"linha {n} está vazia")

print("\n".join(problemas) if problemas else "✅ Cinco linhas, sem 'ataques'.")

Path("notas").mkdir(exist_ok=True)
saida = Path("notas/dia42-resposta-conselho.md")
saida.write_text("# Resposta ao conselho — Dia 42\n\n" + "\n".join(f"{i}. {l}" for i, l in enumerate(linhas, 1))
                 + f"\n\n> {linha6}\n", encoding="utf-8")
print(f"💾 Salvo em {saida}")

## 8. (Opcional) Dados reais, com o filtro ligado

Se quiser repetir com dados reais, ligue `BAIXAR_REAL`. A célula chama a API pública do ransomware.live **para um único grupo**, respeita uma pausa, e passa cada registro pela função `limpar()` **antes** de guardar qualquer coisa. Leia os termos de uso da API antes.

Desligado por padrão: o notebook funciona inteiro sem isso.

In [ ]:
#@title Puxar um grupo real (opcional) { display-mode: "form" }
BAIXAR_REAL = False  #@param {type:"boolean"}
GRUPO_REAL = "akira"  #@param {type:"string"}

import time

if BAIXAR_REAL:
    url = f"https://api.ransomware.live/v2/groupvictims/{GRUPO_REAL}"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "estudo-investigador-cyber"})
        with urllib.request.urlopen(req, timeout=30) as r:
            dados = json.loads(r.read().decode("utf-8"))
        # O filtro roda AQUI, na memória, antes de qualquer gravação.
        reais = [limpar(reg, GRUPO_REAL) for reg in dados]
        del dados                                    # descarta o bruto (com nomes) imediatamente
        with open("postings-real.jsonl", "w", encoding="utf-8") as f:
            for r_ in reais:
                f.write(json.dumps(r_) + "\n")
        print(f"{len(reais)} postagens de {GRUPO_REAL} gravadas SEM campos de vítima.")
        time.sleep(5)                                # educação com a API, como no original
    except Exception as erro:
        print("Não foi possível baixar (sem rede ou limite da API):", type(erro).__name__)
else:
    print("BAIXAR_REAL = False → pulando. Os exercícios acima já cobrem o dia.")

## ✅ Antes de fechar o Dia 42

- [ ] A checagem de campos proibidos imprimiu `0` para todos
- [ ] Você sabe dizer o que acontece com o `grupo-delta` e o `grupo-eco`, e o que o gráfico **não** diz sobre eles
- [ ] Quiz com pelo menos 8/10
- [ ] `notas/dia42-resposta-conselho.md` gerado, sem "ataques"
- [ ] (Original) mapas de calor reais com pandas, anotados com a Operação Cronos e com a conferência no RansomLook

**Amanhã (Dia 43):** o IscaKit vira dado estruturado. Você escreve o cluster do Pescador em **STIX 2.1**, com confiança e TLP dentro de cada objeto.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>